In [ ]:
import pandas as pd

ama = pd.read_excel(r"amocrm_export_leads_2026-02-13.xlsx")

onec = pd.read_excel(r"Ю-АР-СИ.XLSX")
onec = onec[onec['Ответственный'] == 'Яшихин Алексей Валерьевич']

In [ ]:
ama['Дата создания AMO'] = pd.to_datetime(ama['Дата создания'], dayfirst=True)
onec['Дата закрытия 1C'] = pd.to_datetime(onec['Дата выезда'], dayfirst=True)
onec['Дата создания 1C'] = pd.to_datetime(onec['Дата'], dayfirst=True)

C:\Users\a.vorona\AppData\Local\Temp\ipykernel_3292\3574478231.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ama['Дата создания AMA'] = pd.to_datetime(ama['Дата создания'], dayfirst=True)


ЕСли заказ создан в 1С, а в AMA на следующий день (24ч)

In [ ]:
merged = ama.merge(onec, how='cross')
merged['weekday'] = merged['Дата создания AMA'].dt.weekday
merged['Начало 1С - Начало AMA (ч)'] = round((merged['Дата создания 1C'] - merged['Дата создания AMA']).dt.total_seconds() / 3600, 2)

result = merged[
    ((merged['weekday'] <= 4) & (merged['Начало 1С - Начало AMA (ч)'] <= 72) & (merged['Начало 1С - Начало AMA (ч)'] >= 0)) | 
    ((merged['weekday'] == 5) & (merged['Начало 1С - Начало AMA (ч)'] <= 120) & (merged['Начало 1С - Начало AMA (ч)'] >= 0)) 

]

result['Конец 1C - Начало 1C (дни)'] = (
    (result['Дата закрытия 1C'].dt.normalize() - result['Дата создания 1C'].dt.normalize()).dt.days
)


C:\Users\a.vorona\AppData\Local\Temp\ipykernel_3292\990014731.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged = ama.merge(onec, how='cross')
C:\Users\a.vorona\AppData\Local\Temp\ipykernel_3292\990014731.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged['weekday'] = merged['Дата создания AMA'].dt.weekday
C:\Users\a.vorona\AppData\Local\Temp\ipykernel_3292\990014731.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

In [76]:
result.loc[:,['Номер', 'Дата создания AMA', 'Дата создания 1C', 'Дата закрытия 1C', 'Начало 1С - Начало AMA (ч)', 'Конец 1C - Начало 1C (дни)', 'Лот_y']]

,Номер,Дата создания AMA,Дата создания 1C,Дата закрытия 1C,Начало 1С - Начало AMA (ч),Конец 1C - Начало 1C (дни),Лот_y
73,БТС00002755,2026-02-10 22:49:36,2026-02-11 15:32:55,NaT,16.72,NaN,E 04743
74,БТС00002758,2026-02-10 22:49:36,2026-02-11 15:42:25,NaT,16.88,NaN,E 04740
218,БТС00002117,2026-02-04 11:26:10,2026-02-04 11:59:36,2026-02-04,0.56,0.0,B 02105
219,БТС00002182,2026-02-04 11:26:10,2026-02-04 16:43:51,NaT,5.29,NaN,B 02105
220,БТС00002219,2026-02-04 11:26:10,2026-02-05 09:41:43,2026-02-03,22.26,-2.0,B 01559
...,...,...,...,...,...,...,...
4131,БТС00014103,2025-09-15 17:03:47,2025-09-17 12:07:21,2025-09-23,43.06,6.0,B 02106
4202,БТС00012833,2025-08-29 09:53:31,2025-08-29 11:19:35,2025-08-29,1.43,0.0,B 02106
4276,БТС00012654,2025-08-27 10:47:05,2025-08-27 11:13:43,2025-09-15,0.44,19.0,B 01101
4277,БТС00012833,2025-08-27 10:47:05,2025-08-29 11:19:35,2025-08-29,48.54,0.0,B 02106


In [77]:
result = result.loc[(result['Конец 1C - Начало 1C (дни)'] >= 0) |(result['Дата закрытия 1C'].isna()),
           ['Номер', 'Дата создания AMA', 'Дата создания 1C', 
            'Дата закрытия 1C', 'Начало 1С - Начало AMA (ч)', 'Конец 1C - Начало 1C (дни)', 'VIN_Х. (ПР)', 'Лот_y']]

In [78]:
result

,Номер,Дата создания AMA,Дата создания 1C,Дата закрытия 1C,Начало 1С - Начало AMA (ч),Конец 1C - Начало 1C (дни),VIN_Х. (ПР),Лот_y
73,БТС00002755,2026-02-10 22:49:36,2026-02-11 15:32:55,NaT,16.72,NaN,B 01361,E 04743
74,БТС00002758,2026-02-10 22:49:36,2026-02-11 15:42:25,NaT,16.88,NaN,B 01361,E 04740
218,БТС00002117,2026-02-04 11:26:10,2026-02-04 11:59:36,2026-02-04,0.56,0.0,B 02105,B 02105
219,БТС00002182,2026-02-04 11:26:10,2026-02-04 16:43:51,NaT,5.29,NaN,B 02105,B 02105
288,БТС00001770,2026-01-30 13:15:52,2026-01-30 16:11:49,2026-02-04,2.93,5.0,E 04740,B 01562
...,...,...,...,...,...,...,...,...
4131,БТС00014103,2025-09-15 17:03:47,2025-09-17 12:07:21,2025-09-23,43.06,6.0,B 02105,B 02106
4202,БТС00012833,2025-08-29 09:53:31,2025-08-29 11:19:35,2025-08-29,1.43,0.0,B 02106,B 02106
4276,БТС00012654,2025-08-27 10:47:05,2025-08-27 11:13:43,2025-09-15,0.44,19.0,B 01101,B 01101
4277,БТС00012833,2025-08-27 10:47:05,2025-08-29 11:19:35,2025-08-29,48.54,0.0,B 01101,B 02106


In [79]:
df = result[result['Лот_y'] == result['VIN_Х. (ПР)']]
df_all = df.sort_values(by='Дата создания AMA', ascending=True)
display(df_all)
df_all.shape[0]

,Номер,Дата создания AMA,Дата создания 1C,Дата закрытия 1C,Начало 1С - Начало AMA (ч),Конец 1C - Начало 1C (дни),VIN_Х. (ПР),Лот_y
4350,БТС00012300,2025-08-21 14:32:35,2025-08-21 17:48:25,2025-08-25,3.26,4.0,B 02106,B 02106
4276,БТС00012654,2025-08-27 10:47:05,2025-08-27 11:13:43,2025-09-15,0.44,19.0,B 01101,B 01101
4202,БТС00012833,2025-08-29 09:53:31,2025-08-29 11:19:35,2025-08-29,1.43,0.0,B 02106,B 02106
4130,БТС00014102,2025-09-15 17:03:47,2025-09-17 12:05:41,2025-09-24,43.03,7.0,B 02105,B 02105
4129,БТС00014033,2025-09-15 17:03:47,2025-09-16 15:38:39,NaT,22.58,NaN,B 02105,B 02105
4056,БТС00014103,2025-09-15 17:04:23,2025-09-17 12:07:21,2025-09-23,43.05,6.0,B 02106,B 02106
3978,БТС00013979,2025-09-15 17:04:49,2025-09-16 10:08:58,2025-09-24,17.07,8.0,B 01104,B 01104
3759,БТС00015977,2025-10-13 08:21:52,2025-10-13 08:54:30,2025-10-21,0.54,8.0,B 02106,B 02106
3389,БТС00017124,2025-10-27 16:32:56,2025-10-28 09:47:40,2025-10-28,17.25,0.0,B 01361,B 01361
3317,БТС00017597,2025-11-03 10:41:06,2025-11-05 08:23:23,2025-11-05,45.70,0.0,B 02105,B 02105


35

In [81]:
df_all.to_excel('result_new.xlsx', index=False)